vector `pylogprob` $= \left[\ln(f(x_1)),\ln(f(x_2)),\ldots\right]$

vector `py_logprob` $= \left[\ln(g(x_1)),\ln(g(x_2)),\ldots\right]$

In [ ]:
pylogprob = tf.expand_dims(py.distribution.log_prob(y_batch),1)
py_logprob = tf.expand_dims(py_.distribution.log_prob(y_batch),1)

vector `logmax` $= \left[\max\left(\ln(f(x_1,)),\ln(g(x_1))\right),\max\left(\ln(f(x_1)),\ln(g(x_1))\right),\ldots\right]$

In [ ]:
logmax = tf.stop_gradient(tf.math.maximum(pylogprob,py_logprob)+0.000001)
logmax = tf.constant(-math.log(0.9999)) # overwritten

$c =$ `logmax` vector

$l_1 =$ `pylogprob`, $l_2 =$ `py_logprob` vectors

vector `alpha` $= \alpha = \ln\left(e^{l_1 - c} + e^{l_2 - c}\right) - \ln(2) = \ln\left(\frac{e^{l_1} + e^{l_2}}{2}\right) - c = \ln\left(\frac{\frac{f(x_1) + f(x_2)}{2}}{\max\left(f(x_{1}),f(x_{2})\right)}\right)$

In [ ]:
logmean_logmax = tf.math.reduce_logsumexp(tf.concat([pylogprob-logmax,py_logprob-logmax], 1),axis=1) - tf.log(2.)
alpha = tf.expand_dims(logmean_logmax,1) # makes column vector

vector `hmax` $= h = 2 \left(\frac{\alpha}{(1 - e^\alpha)^2} + \frac{1}{e^\alpha(1 - e^\alpha)}\right)$

In [ ]:
if (algorithm==3):
    hmax = 2*tf.stop_gradient(alpha/tf.math.pow(1-tf.math.exp(alpha),2) + tf.math.pow(tf.math.exp(alpha)*(1-tf.math.exp(alpha)),-1))
else:
    hmax=1.

`var` $= \frac{1}{2} \left(\mathbb{E}(e^{2 (l_1 -  c)} \cdot h) - \mathbb{E}(e^{l_1 + l_2 - 2 c} \cdot h)\right)$
$= \frac{1}{2} \left(\mathbb{E}\left(\frac{f^2(x)}{\max^2(f(x),g(x))}h\right) - \mathbb{E}\Big(\frac{f(x)g(x)}{\max^2(f(x),g(x))}h\Big)\right)$

In [ ]:
var = 0.5*(tf.reduce_mean(tf.exp(2*pylogprob-2*logmax)*hmax) - tf.reduce_mean(tf.exp(pylogprob + py_logprob - 2*logmax)*hmax))

`datalikelihood` $= \mathbb{E}\Big(\ln\big(f(x)\big)\Big)$

In [ ]:
datalikelihood = tf.reduce_mean(py.distribution.log_prob(y_batch))

`logprior` $=\sum p_W(v) + \sum p_b(v) + \sum p_{W_{\text{out}}}(v) + \sum p_{b_{\text{out}}}(v)$

`entropy` $= - \sum q_W(v) - \sum q_b(v) - \sum q_{W_{\text{out}}}(v) - \sum q_{b_{\text{out}}}(v)$

`KL` $= \frac{\sum q_W(v) + \sum q_b(v) + \sum q_{W_{\text{out}}}(v) + \sum q_{b_{\text{out}}}(v) - \sum p_W(v) - \sum p_b(v) - \sum p_{W_{\text{out}}}(v) - \sum p_{b_{\text{out}}}(v)}{N}$

In [ ]:
logprior = tf.reduce_sum(pW.distribution.log_prob(pW.value)) + \
            tf.reduce_sum(pb.distribution.log_prob(pb.value)) + \
            tf.reduce_sum(pW_out.distribution.log_prob(pW_out.value)) + \
            tf.reduce_sum(pb_out.distribution.log_prob(pb_out.value))

entropy = tf.reduce_sum(qW.distribution.log_prob(qW.value)) + \
            tf.reduce_sum(qb.distribution.log_prob(qb.value)) + \
            tf.reduce_sum(qW_out.distribution.log_prob(qW_out.value)) + \
            tf.reduce_sum(qb_out.distribution.log_prob(qb_out.value))

entropy = -entropy

KL = (- entropy - logprior)/N

In [ ]:
if algorithm==2 or algorithm==3:
    elbo = datalikelihood + var - KL
elif algorithm==1:
    elbo = datalikelihood - KL
elif algorithm==0:
    elbo = datalikelihood + logprior/N

verbose=True
optimizer = tf.train.AdamOptimizer(0.001)
t = []
train = optimizer.minimize(-elbo)